# 얼굴가드 촬영 열화 강건성 평가 — Kaggle 무료 GPU

쉽게 말하면 **깨끗한 영상에서 등록한 얼굴이 압축·흐림·어두움·저해상도에서도 같은 사람으로 인식되는지** 확인하는 노트북이다.

## 실행 전 딱 세 가지

1. 오른쪽 `Settings`에서 `Accelerator`를 `GPU P100`으로 선택한다.
2. 오른쪽 `Input`에서 승인받은 929MB ZIP을 담은 **비공개 Kaggle Dataset**을 연결한다.
3. `Internet`을 켠 뒤 아래 셀을 위에서부터 실행한다. 모델 파일 최초 다운로드에만 필요하다.

원본 영상·얼굴 이미지·사람별 점수·임베딩은 `/kaggle/temp`에서만 처리한다. 최종 `/kaggle/working`에는 사람을 식별할 수 없는 집계 결과 ZIP만 남긴다. 이 실험은 얼굴 동일인 검증이며 딥페이크 탐지 정확도가 아니다.

In [ ]:
# 1. 실행 설정과 이용 조건 확인
REPO_URL = "https://github.com/Chunbae-A/face-image.git"
BRANCH = "exp/6-faceguard-robustness"
CODE_SOURCE = "embedded"  # "embedded" 권장, 필요하면 "github"

# 비워두면 /kaggle/input 아래에서 정확한 크기의 Celeb-DF-v2.zip을 자동으로 찾는다.
SOURCE_ZIP_PATH = ""
EXPECTED_SOURCE_ZIP_BYTES = 928989923
# Kaggle Dataset이 ZIP을 자동으로 풀었을 때 검증할 590개 MP4의 정확한 합계다.
EXPECTED_EXTRACTED_VIDEO_BYTES = 946501150

# Celeb-DF 이용 조건에서 Kaggle 비공개 데이터 처리도 허용되는지 확인한 경우만 True.
I_CONFIRM_CELEBDF_KAGGLE_PRIVATE_PROCESSING_IS_ALLOWED = False
# InsightFace buffalo_l 가중치는 비상업 연구 전용이다.
I_ACCEPT_INSIGHTFACE_NONCOMMERCIAL_RESEARCH_LICENSE = False

FRAMES_PER_VIDEO = 5
MINIMUM_VALID_FRAMES = 3
CONDITIONS = (
    "clean",
    "jpeg_q30",
    "gaussian_blur_sigma2",
    "low_light_gamma2",
    "downscale_0_25",
    "combined_mobile_stress",
)
SEEDS = (20260805, 20260806, 20260807, 20260808, 20260809)
BOOTSTRAP_REPEATS = 500
RUN_SMOKE_BEFORE_FULL = True

import os
from pathlib import Path
import sys

IN_KAGGLE = Path("/kaggle").exists() and bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
if IN_KAGGLE and not I_CONFIRM_CELEBDF_KAGGLE_PRIVATE_PROCESSING_IS_ALLOWED:
    raise PermissionError("Celeb-DF의 Kaggle 비공개 처리 허용 여부를 먼저 확인하세요.")
if not I_ACCEPT_INSIGHTFACE_NONCOMMERCIAL_RESEARCH_LICENSE:
    raise PermissionError("InsightFace 비상업 연구용 가중치 조건을 확인하세요.")
if EXPECTED_SOURCE_ZIP_BYTES <= 0:
    raise ValueError("승인 ZIP의 정확한 바이트를 입력해야 합니다.")

print({
    "kaggle": IN_KAGGLE,
    "frames_per_video": FRAMES_PER_VIDEO,
    "conditions": CONDITIONS,
    "seeds": SEEDS,
    "maximum_frame_inferences": 590 * FRAMES_PER_VIDEO * len(CONDITIONS),
})

## 실행 환경

Kaggle 오른쪽 설정에서 GPU와 Internet을 켠다. 이 노트북은 민감한 중간 산출물을 저장 결과에 포함하지 않도록 `/kaggle/temp`와 `/kaggle/working`을 분리한다.

In [ ]:
# 2. 라이브러리 설치
%pip uninstall -y -q onnxruntime onnxruntime-gpu
%pip install -q --no-cache-dir "insightface==1.0.1" "onnxruntime-gpu==1.23.2" opencv-python-headless pandas matplotlib seaborn tqdm

In [ ]:
# 3. 실행 코드 준비 — GitHub 권한이 필요 없는 내장 코드가 기본값
from pathlib import Path
import base64
import os
import subprocess

EMBEDDED_FILES_B64 = {'scripts/celebdf_faceguard.py': '...', 'scripts/run_celebdf_arcface.py': '...', 'scripts/audit_celebdf_robustness.py': '...'}
EMBEDDED_CODE_SHA256 = "be50c18a7a6a0d2047daad61765ea2ad986ad69c1e248e2cf9ee20fd29c03875"

if IN_KAGGLE and CODE_SOURCE == "github":
    REPO_DIR = Path("/kaggle/working/face-image")
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "checkout", BRANCH], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    CODE_VERSION = subprocess.check_output(
        ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
    ).strip()
elif IN_KAGGLE:
    REPO_DIR = Path("/kaggle/working/face-image")
    for relative_path, encoded in EMBEDDED_FILES_B64.items():
        target = REPO_DIR / relative_path
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(base64.b64decode(encoded))
    CODE_VERSION = f"embedded:{EMBEDDED_CODE_SHA256[:12]}"
else:
    REPO_DIR = Path.cwd()
    try:
        CODE_VERSION = subprocess.check_output(
            ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
        ).strip()
    except (FileNotFoundError, subprocess.SubprocessError):
        CODE_VERSION = f"local:{EMBEDDED_CODE_SHA256[:12]}"

os.chdir(REPO_DIR)
print({"repo": str(REPO_DIR), "code_source": CODE_SOURCE, "code_version": CODE_VERSION})

In [ ]:
# 4. 비공개 Kaggle Dataset 원본 확인
import json
import shutil
import zipfile

# /kaggle/temp는 Save Version 결과에 포함하지 않는 민감정보 임시 처리 영역이다.
WORK_ROOT = Path("/kaggle/temp/celebdf_robustness") if IN_KAGGLE else REPO_DIR / "outputs" / "celebdf_robustness"
WORK_ROOT.mkdir(parents=True, exist_ok=True)

input_mode = "zip"
source_input = None
runtime_zip = None

if IN_KAGGLE:
    if SOURCE_ZIP_PATH.strip():
        zip_candidates = [Path(SOURCE_ZIP_PATH).expanduser()]
    else:
        zip_candidates = sorted(Path("/kaggle/input").rglob("Celeb-DF-v2.zip"))
    exact_zips = [
        path for path in zip_candidates
        if path.is_file() and path.stat().st_size == EXPECTED_SOURCE_ZIP_BYTES
    ]

    if len(exact_zips) == 1:
        source_input = exact_zips[0]
        runtime_zip = exact_zips[0]
    elif exact_zips:
        raise FileExistsError(f"정확한 Celeb-DF ZIP이 여러 개입니다: {exact_zips}")
    else:
        # Kaggle은 업로드한 ZIP을 자동으로 풀 수 있다. 이 경우 590개 파일 수와
        # 정확한 총 바이트를 확인한 뒤 /kaggle/temp에 저장 방식 ZIP을 재구성한다.
        celeb_real_dirs = sorted({
            path.parent
            for path in Path("/kaggle/input").rglob("Celeb-real/*.mp4")
            if path.is_file()
        })
        exact_dirs = []
        for directory in celeb_real_dirs:
            videos = sorted(directory.glob("*.mp4"))
            total_bytes = sum(path.stat().st_size for path in videos)
            if len(videos) == 590 and total_bytes == EXPECTED_EXTRACTED_VIDEO_BYTES:
                exact_dirs.append((directory, videos))
        if len(exact_dirs) != 1:
            found = [
                {
                    "directory": str(directory),
                    "video_count": len(videos),
                    "total_bytes": sum(path.stat().st_size for path in videos),
                }
                for directory, videos in exact_dirs
            ]
            raise FileNotFoundError(
                "정확한 Celeb-real MP4 590개 묶음 하나를 찾지 못했습니다. "
                f"Kaggle Input 연결을 확인하세요. 일치={found}"
            )

        celeb_real_dir, videos = exact_dirs[0]
        source_input = celeb_real_dir
        input_mode = "kaggle_auto_extracted_mp4"
        repacked_dir = WORK_ROOT / "source_repacked"
        repacked_dir.mkdir(parents=True, exist_ok=True)
        runtime_zip = repacked_dir / "Celeb-DF-v2.zip"
        partial_zip = runtime_zip.with_suffix(".zip.partial")
        if not runtime_zip.exists():
            partial_zip.unlink(missing_ok=True)
            with zipfile.ZipFile(partial_zip, "w", compression=zipfile.ZIP_STORED) as archive:
                for video in videos:
                    archive.write(video, arcname=f"Celeb-real/{video.name}")
            partial_zip.replace(runtime_zip)

        with zipfile.ZipFile(runtime_zip) as archive:
            repacked_members = [
                item for item in archive.infolist()
                if not item.is_dir() and item.filename.startswith("Celeb-real/")
            ]
        if len(repacked_members) != 590:
            raise IOError(f"재구성 ZIP 영상 수가 다릅니다: {len(repacked_members)} != 590")
else:
    source_input = Path(SOURCE_ZIP_PATH).expanduser()
    if not source_input.is_file():
        raise FileNotFoundError(f"Celeb-DF ZIP을 찾지 못했습니다: {source_input}")
    if source_input.stat().st_size != EXPECTED_SOURCE_ZIP_BYTES:
        raise IOError(
            f"Celeb-DF ZIP 크기가 다릅니다: {source_input.stat().st_size} "
            f"!= {EXPECTED_SOURCE_ZIP_BYTES}"
        )
    runtime_zip = source_input

assert source_input is not None
assert runtime_zip is not None

VIDEO_ROOT = WORK_ROOT / "videos"
MANIFEST = WORK_ROOT / "celeb_real_manifest.csv"
INVENTORY_JSON = WORK_ROOT / "celeb_real_inventory.json"
CONDITION_ROOT = WORK_ROOT / "conditions"
SANITIZED_ROOT = WORK_ROOT / "sanitized"
for path in (VIDEO_ROOT, CONDITION_ROOT, SANITIZED_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print({
    "input_mode": input_mode,
    "source_input": str(source_input),
    "runtime_zip_gb": round(runtime_zip.stat().st_size / 1e9, 3),
    "private_work_root": str(WORK_ROOT),
    "runtime_free_gb": round(shutil.disk_usage(WORK_ROOT).free / 1e9, 2),
})

In [ ]:
# 5. Celeb-real 590개 확인과 추출
subprocess.run([
    sys.executable, "scripts/celebdf_faceguard.py", "inventory", str(runtime_zip),
    "--manifest", str(MANIFEST), "--summary", str(INVENTORY_JSON),
], check=True)
inventory = json.loads(INVENTORY_JSON.read_text(encoding="utf-8"))
assert inventory["video_count"] == 590, inventory
assert inventory["subject_count"] == 59, inventory
assert inventory["eligible_subjects_ge_8_videos"] == 56, inventory

subprocess.run([
    sys.executable, "scripts/celebdf_faceguard.py", "extract", str(runtime_zip),
    "--manifest", str(MANIFEST), "--output", str(VIDEO_ROOT), "--mode", "full",
], check=True)
extracted = sorted((VIDEO_ROOT / "Celeb-real").glob("*.mp4"))
if len(extracted) != 590:
    raise RuntimeError(f"590개 영상이 필요하지만 {len(extracted)}개를 찾았습니다.")
print({"videos": len(extracted), "eligible_subjects": 56})

In [ ]:
# 6. GPU와 ONNX Runtime 확인
import onnxruntime as ort

providers = ort.get_available_providers()
print({"onnxruntime": ort.__version__, "providers": providers})
if IN_KAGGLE and "CUDAExecutionProvider" not in providers:
    raise RuntimeError(
        "Kaggle GPU가 연결되지 않았습니다. 오른쪽 Settings에서 Accelerator를 GPU로 바꾸고 세션을 재시작하세요."
    )
print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    text=True,
))

In [ ]:
# 7. 깨끗한 영상과 다섯 가지 촬영 열화 조건 추론
EMBEDDING_RUNS = {}
RUN_REPORTS = {}
REJECT_FILES = {}

for condition in CONDITIONS:
    run_root = CONDITION_ROOT / condition
    run_root.mkdir(parents=True, exist_ok=True)
    embeddings = run_root / "video_embeddings.npz"
    rejects = run_root / "rejects.csv"
    run_report = run_root / "run.json"
    command = [
        sys.executable, "scripts/run_celebdf_arcface.py",
        "--manifest", str(MANIFEST),
        "--video-root", str(VIDEO_ROOT),
        "--output", str(embeddings),
        "--rejects", str(rejects),
        "--run-report", str(run_report),
        "--frames-per-video", str(FRAMES_PER_VIDEO),
        "--minimum-valid-frames", str(MINIMUM_VALID_FRAMES),
        "--input-condition", condition,
        "--checkpoint-every", "25",
        "--progress-every", "25",
        "--model-name", "buffalo_l",
        "--accept-noncommercial-model-license",
    ]
    if RUN_SMOKE_BEFORE_FULL and condition == "clean" and not embeddings.exists():
        subprocess.run(
            command + [
                "--mode", "smoke", "--smoke-subjects", "2",
                "--smoke-videos-per-subject", "1",
            ],
            check=True,
        )
    subprocess.run(command + ["--mode", "full"], check=True)
    completed = json.loads(run_report.read_text(encoding="utf-8"))
    if completed["status"] != "completed" or completed["input_condition"] != condition:
        raise RuntimeError(f"조건 실행이 완료되지 않았습니다: {condition}")
    EMBEDDING_RUNS[condition] = embeddings
    RUN_REPORTS[condition] = run_report
    REJECT_FILES[condition] = rejects
    print({
        "completed_condition": condition,
        "successful_videos": completed["successful_video_count_total"],
    })

print({
    "completed_conditions": tuple(EMBEDDING_RUNS),
    "embeddings_stay_in_kaggle_temp": True,
})

In [ ]:
# 8. 공통 query 평가와 판정 기준값 보정
audit_command = [
    sys.executable, "scripts/audit_celebdf_robustness.py",
    "--output-dir", str(SANITIZED_ROOT),
    "--seeds", ",".join(str(seed) for seed in SEEDS),
    "--reference-count", "3",
    "--bootstrap-repeats", str(BOOTSTRAP_REPEATS),
]
for condition in CONDITIONS:
    audit_command.extend(["--embedding-run", f"{condition}={EMBEDDING_RUNS[condition]}"])
    audit_command.extend(["--run-report", f"{condition}={RUN_REPORTS[condition]}"])
    if REJECT_FILES[condition].exists():
        audit_command.extend(["--rejects", f"{condition}={REJECT_FILES[condition]}"])
subprocess.run(audit_command, check=True)

AUDIT_JSON = SANITIZED_ROOT / "celebdf_robustness_audit.json"
METRICS_CSV = SANITIZED_ROOT / "celebdf_robustness_metrics.csv"
SUMMARY_CSV = SANITIZED_ROOT / "celebdf_robustness_summary.csv"
audit_report = json.loads(AUDIT_JSON.read_text(encoding="utf-8"))
assert all(item["validation_test_identity_overlap"] == 0 for item in audit_report["leakage_checks"])
assert all(item["registration_query_video_overlap"] == 0 for item in audit_report["leakage_checks"])
print(json.dumps(audit_report["decisions"], ensure_ascii=False, indent=2))

In [ ]:
# 9. 조건별 결과 표 확인
import pandas as pd

summary = pd.read_csv(SUMMARY_CSV)
display(summary[[
    "condition",
    "test_roc_auc_mean",
    "test_eer_mean",
    "far_0.001_clean_locked_test_tar_mean",
    "far_0.001_clean_locked_test_far_mean",
    "far_0.001_condition_calibrated_test_tar_mean",
    "far_0.001_condition_calibrated_test_far_mean",
    "far_0.001_threshold_shift_mean",
]])
display(pd.DataFrame([
    {
        "condition": item["condition"],
        **item["quality"],
        "reject_reason_counts": item["reject_reason_counts"],
    }
    for item in audit_report["input_runs"]
]))

In [ ]:
# 10. 촬영 열화별 성능 그래프 생성
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
order = list(CONDITIONS)
figure, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.barplot(
    data=summary,
    x="condition",
    y="far_0.001_clean_locked_test_tar_mean",
    order=order,
    color="#4c78a8",
    ax=axes[0],
)
axes[0].set(title="Recognition rate with clean-locked threshold", ylabel="Test TAR", xlabel="")
axes[0].tick_params(axis="x", rotation=35)
sns.barplot(
    data=summary,
    x="condition",
    y="far_0.001_clean_locked_test_far_mean",
    order=order,
    color="#f58518",
    ax=axes[1],
)
axes[1].axhline(0.001, color="red", linestyle="--", linewidth=1, label="target FAR")
axes[1].set(title="False acceptance with clean-locked threshold", ylabel="Test FAR", xlabel="")
axes[1].tick_params(axis="x", rotation=35)
axes[1].legend()
figure.tight_layout()
FIGURE_PNG = SANITIZED_ROOT / "celebdf_robustness_audit.png"
figure.savefig(FIGURE_PNG, dpi=180, bbox_inches="tight")
plt.show()

In [ ]:
# 11. 얼굴 없는 집계 결과만 Kaggle Output에 저장
import hashlib
import zipfile

RUNTIME_CONFIG = SANITIZED_ROOT / "robustness_runtime_config.json"
RUNTIME_CONFIG.write_text(json.dumps({
    "environment": "kaggle",
    "code_version": CODE_VERSION,
    "source_zip_bytes": EXPECTED_SOURCE_ZIP_BYTES,
    "frames_per_video": FRAMES_PER_VIDEO,
    "minimum_valid_frames": MINIMUM_VALID_FRAMES,
    "conditions": CONDITIONS,
    "reference_count": 3,
    "reserved_registration_count": 5,
    "seeds": SEEDS,
    "bootstrap_repeats": BOOTSTRAP_REPEATS,
    "raw_data_in_bundle": False,
    "embeddings_in_bundle": False,
}, ensure_ascii=False, indent=2), encoding="utf-8")

temporary_bundle = SANITIZED_ROOT / "celebdf_robustness_results.zip"
allowed_artifacts = (AUDIT_JSON, METRICS_CSV, SUMMARY_CSV, FIGURE_PNG, RUNTIME_CONFIG)
with zipfile.ZipFile(temporary_bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in allowed_artifacts:
        archive.write(path, arcname=path.name)

with zipfile.ZipFile(temporary_bundle) as archive:
    names = set(archive.namelist())
    if names != {path.name for path in allowed_artifacts}:
        raise AssertionError(f"결과 ZIP 허용 목록이 다릅니다: {sorted(names)}")
    if any(name.endswith(".npz") for name in names):
        raise AssertionError("결과 ZIP에 임베딩이 포함되었습니다.")

RESULT_BUNDLE = Path("/kaggle/working/celebdf_robustness_results.zip") if IN_KAGGLE else temporary_bundle
if IN_KAGGLE:
    shutil.copy2(temporary_bundle, RESULT_BUNDLE)
bundle_sha256 = hashlib.sha256(RESULT_BUNDLE.read_bytes()).hexdigest()

# 원본에서 파생된 영상, ID manifest, 실패 ID, 임베딩을 모두 지운다.
if IN_KAGGLE:
    shutil.rmtree(WORK_ROOT)
    forbidden_suffixes = {".mp4", ".npz"}
    leaked = [
        str(path) for path in Path("/kaggle/working").rglob("*")
        if path.is_file() and path.suffix.lower() in forbidden_suffixes
    ]
    if leaked:
        raise AssertionError(f"Kaggle Output에 민감한 중간 파일이 남았습니다: {leaked}")

print({
    "result_bundle": str(RESULT_BUNDLE),
    "bundle_sha256": bundle_sha256,
    "size_mb": round(RESULT_BUNDLE.stat().st_size / 1e6, 2),
    "private_work_root_deleted": IN_KAGGLE,
    "raw_or_embedding_files_in_bundle": False,
})

## 완료 후 해야 할 일

1. 마지막 셀에 `private_work_root_deleted: True`가 표시됐는지 확인한다.
2. 오른쪽 `Output` 또는 실행 결과에서 `celebdf_robustness_results.zip`만 내려받는다.
3. 원본 영상이나 `.npz` 임베딩은 GitHub에 올리지 않는다.

결과는 얼굴 검출 성공률 → 깨끗한 기준값의 TAR/FAR → 조건별 보정 결과 순서로 읽는다. Celeb-real 결과는 한국인 얼굴·실제 휴대전화·운영 트래픽의 성능을 대신하지 않는다.